<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-11-self-hosting/lesson-11.4-deploy-slm/notebooks/GCP_Capstone_11.4_DeploySLM.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.4 Deploy Your Own Model — The GPU Is a Bill
**Netsetos GenAI Engineering — GCP Capstone** · Module 11 · rebuilt on the live lane, 10 September 2026

Take the model 10.5 tuned, put it behind an L4 on Cloud Run, behind the gateway from 11.3, and behind the API - then work out, in rupees, whether that was worth doing. The bill goes first because it decides the rest: about 379,000 answers a month before the instance beats Gemini, and the region a residency promise needs is the one you cannot self-serve. Then the parts that used to be printed and are now run: the Modelfile generated from the tokenizer and read back from the bucket, the deployed service with its cold start timed, one completion through the gateway's route, the candidate revision on `MODEL_BACKEND=gateway` judged by the same gate as the tuned Gemini, one tenant pinned by a Firestore document and read back off the usage row, the honest comparison live, and the service asserted back at zero.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-firestore==2.30.0 pandas==2.3.3 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
ROWS          = 10      # golden rows the comparison sends to each backend (make compare sends 20)
CANDIDATE_URL = f"https://candidate---documind-api-{NUMBER}.{REGION}.run.app"   # make candidate MODEL_BACKEND=gateway GENERATOR_MODEL=documind-slm

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body, headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers):
    the body's `model` names what answered (a fallback included), the headers carry the cost the gateway priced."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def slm(path: str, body: dict | None = None, method: str = "POST", timeout: int = 240) -> tuple[int, dict | str, float]:
    """One call to the SLM's own doors (Ollama's /api/*, or its OpenAI-compatible /v1/*), timed - the first call after
    idle is the cold start."""
    t0 = time.time()
    r = requests.request(method, f"{SLM_URL}{path}", json=body, timeout=timeout,
                         headers={"Authorization": f"Bearer {documind_tools._id_token(SLM_URL)}"})
    try:
        return r.status_code, r.json(), time.time() - t0
    except ValueError:
        return r.status_code, r.text[:400], time.time() - t0

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: run_eval, judge
sys.path.insert(0, f"{KIT}/deploy/services/slm")        # compare_backends, make_modelfile
print("helpers: api(), gateway(), slm(), service(), usage_rows(); the kit's evals/ and services/slm/ on sys.path")


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


## Cell 2: What an idle GPU costs - and the break-even
Kept from the first version: the four rows, read before quoting any of them.


In [ ]:
# Before you deploy anything: what does an idle GPU cost?
#
# This is the number that decides whether the rest of the lesson is a good idea,
# so it goes first rather than last. And it is NOT the GPU price - that is the
# mistake this section exists to prevent.
USD_INR = 85

# 11.1's table, in full. Read all four rows before quoting any of them.
COMPONENTS = [
    ('L4 GPU (no zonal redundancy)', 0.672),
    ('CPU (8 vCPU)',                 0.518),
    ('Memory (32 GiB)',              0.230),
]
print(f'  {"component":32} {"per hour":>9} {"per month":>11} {"INR/month":>12}')
total = 0.0
for label, rate in COMPONENTS:
    total += rate
    print(f'  {label:32} ${rate:>8.3f} ${rate * 24 * 30:>10,.0f} '
          f'Rs {rate * 24 * 30 * USD_INR:>9,.0f}')
print(f'  {"TOTAL INSTANCE (recommended)":32} ${total:>8.2f} ${total * 24 * 30:>10,.0f} '
      f'Rs {total * 24 * 30 * USD_INR:>9,.0f}')
print()
print('  The GPU is less than HALF the bill. CPU and memory add $0.748/hr on top')
print('  of a $0.672 GPU, and the deploy command in step 5 asks for 8 vCPU and')
print('  32 GiB - so $1.42/hr is the rate that applies, not $0.672.')
print()

L4_MONTH_INR = total * 24 * 30 * USD_INR

# Gemini, for one ordinary DocuMind answer.
GEM_IN, GEM_OUT = 1.50, 7.50     # USD per 1M tokens, standard rates
TOK_IN, TOK_OUT = 800, 200


def gemini_inr(n_answers: int = 1) -> float:
    return (TOK_IN * GEM_IN + TOK_OUT * GEM_OUT) / 1e6 * USD_INR * n_answers


print(f'  gemini-3.6-flash: Rs {gemini_inr():.4f} per answer (800 in / 200 out)')
print(f'  your own instance: Rs {L4_MONTH_INR:,.0f} per MONTH, whatever the volume')
print()
breakeven = L4_MONTH_INR / gemini_inr()
print(f'  BREAK-EVEN: {breakeven:,.0f} answers a month.')
print()
print(f'  {"answers/month":>15} {"gemini":>12} {"self-hosted":>13}  cheaper')
for n in (10_000, 50_000, 379_000, 500_000, 2_000_000):
    g = gemini_inr(n)
    print(f'  {n:>15,} {g:>12,.0f} {L4_MONTH_INR:>13,.0f}  '
          f'{"self-hosted" if L4_MONTH_INR < g else "gemini"}')
print()
print('  Below ~379,000 answers a month, running your own model costs MORE.')
print('  Say that out loud before you build it. The reasons to self-host that')
print('  survive this table are the ones from 10.5 - residency and format - and')
print('  they are worth paying for. "It is cheaper" is only true above the line.')


## Cell 3: The region you want is the one you cannot have


In [ ]:
# The problem with the reason you did all this.
#
# 10.5 fine-tuned for two reasons: FORMAT and RESIDENCY. The bank customer
# needs its answers produced inside India. So deploy the L4 in Mumbai.
#
# You cannot. Lesson 11.1 already established it, and it is worth re-reading
# before you promise anyone anything:
L4_REGIONS = {
    'us-central1':    'GA, self-serve',
    'us-east4':       'GA, self-serve',
    'europe-west1':   'GA, self-serve',
    'europe-west4':   'GA, self-serve',
    'asia-southeast1': 'GA, self-serve  (Singapore - near India, not IN India)',
    'asia-south1':    'INVITATION ONLY  (Mumbai - the one you actually want)',
}
print(f'  {"region":18} L4 availability')
for r, s in L4_REGIONS.items():
    mark = '  <-- ' if 'INVITATION' in s else '      '
    print(f'  {r:18}{mark}{s}')

print()
print('  The obvious region for an Indian workload is the one you cannot')
print('  self-serve. That is not a bug in your plan; it is the plan meeting')
print('  hardware supply.')
print()
# So the residency promise has three honest resolutions, and they are all
# trade-offs rather than fixes:
OPTIONS = [
    ('us-central1 L4',
     'Rs 86,904/mo',
     'cheapest and available - but inference leaves India, which was the point'),
    ('asia-south1 L4 (Mumbai)',
     'Rs 86,904/mo',
     'exactly what you wanted; request access and wait, with no date promised'),
    ('asia-south2 RTX PRO 6000 (Delhi)',
     're-price it',
     'GA and self-serve, IN India - but a 96 GB card is not an L4 at an L4 price'),
]
print(f'  {"option":34} {"cost":>13}  note')
for name, cost, note in OPTIONS:
    print(f'  {name:34} {cost:>13}  {note}')
print()
print('  This lesson deploys to us-central1 because it must run for everyone')
print('  today. If your contract says India, the honest answer is asia-south2')
print('  with a fresh price, or a Mumbai invitation you have actually received -')
print('  NOT us-central1 with the word "residency" in the proposal.')
print()
print('  Say this to the customer before you build it, not after. A residency')
print('  claim that depends on a GPU allocation you do not have is a promise')
print('  someone else has to keep.')


## Cell 4: The Modelfile comes from the tokenizer
Read back from the bucket beside the GGUF; the committed placeholder shown as the bug it would be.


In [ ]:
from make_modelfile import build

# THE MODELFILE COMES FROM THE TOKENIZER. The GGUF carries the weights and nothing else; the chat template and the stop
# tokens are the runtime's, and a template typed from a blog post is the commonest way a working fine-tune looks
# broken - fluent text that never stops. 10.5 generates it with make_modelfile.py from the checkpoint's own tokenizer
# and uploads it beside the GGUF; make deploy-slm stages both. The committed Modelfile is a placeholder whose TEMPLATE
# is the bare prompt, shown here as the bug it would be if an image were ever built from it.
placeholder = open(f"{KIT}/deploy/services/slm/Modelfile", encoding="utf-8").read()
tmpl = placeholder.split('TEMPLATE """', 1)[1].split('"""', 1)[0]
print("the committed placeholder's TEMPLATE:", repr(tmpl), "- no turn markers, no stop tokens: a Gemma answers that and never stops")
assert tmpl == "{{ .Prompt }}", "the placeholder grew a template: it must stay a placeholder; the real one is generated"
gguf = gcs.bucket(DATASETS).blob("sft/documind-slm.gguf")
mf = gcs.bucket(DATASETS).blob("sft/documind-slm.Modelfile")
if mf.exists() and gguf.exists():
    gguf.reload()
    real = mf.download_as_text()
    print(f"\ngs://{DATASETS}/sft/documind-slm.gguf: {gguf.size / 1e9:.2f} GB, and the Modelfile 10.5 generated beside it:\n")
    print(chr(10).join(l for l in real.splitlines() if l.startswith(("FROM", "TEMPLATE", "PARAMETER"))))
    assert "PARAMETER stop" in real and "{{ .Prompt }}" in real, "the generated Modelfile must carry the tokenizer's stop tokens"
    SLM_SOURCE_EXPECTED = "gguf"
else:
    print(f"\n10.5's T4 pass has not landed in gs://{DATASETS}/sft/ yet. Offline, build() shows the shape on a Gemma-style template:\n")
    print(build("<start_of_turn>user\n{{ .Prompt }}<end_of_turn>\n<start_of_turn>model\n", ["<eos>", "<end_of_turn>"], "documind-slm.gguf"))
    print(f"until then: make deploy-slm PROJECT={PROJECT_ID} SLM_STOCK=gemma3:4b builds the stand-in, and the service's slm-source label says so")
    SLM_SOURCE_EXPECTED = "stock"


## Cell 5: The image and the deploy, from the kit
The Dockerfile's three load-bearing lines and `make deploy-slm` flag by flag; then where a cold start's seconds go.


In [ ]:
# THE IMAGE AND THE DEPLOY, FROM THE KIT. Three load-bearing lines in the Dockerfile: the Ollama pin (0.33.3 - the
# 0.33 line is where gemma4 arrived, and an older Ollama loads a Gemma 4 GGUF and produces nonsense rather than
# refusing), OLLAMA_HOST on 8080 (Cloud Run's $PORT; the default 11434 fails the probe with no useful log line), and
# `ollama create` at BUILD time (importing a 2 GB GGUF is time a cold start does not have). Then make deploy-slm flag
# by flag: each is a bill or a failure mode, and the probe waits for the model, not the process.
df = open(f"{KIT}/deploy/services/slm/Dockerfile", encoding="utf-8").read()
for needle in ("FROM ollama/ollama:0.33.3", "ENV OLLAMA_HOST=0.0.0.0:8080", "COPY build/ /build/", "ollama create documind-slm -f /build/Modelfile"):
    assert needle in df, needle
print(chr(10).join(l for l in df.splitlines() if l and not l.startswith("#")))
mk = open(f"{KIT}/deploy/Makefile", encoding="utf-8").read()
DEPLOY_SLM = mk.split("deploy-slm:", 1)[1].split(chr(10) * 2, 1)[0]
print()
print(chr(10).join(l for l in DEPLOY_SLM.splitlines() if l.strip().startswith(("gcloud run deploy", "--"))))
FLAGS = [("--gpu 1 --gpu-type nvidia-l4", "one L4, lowercase l; nvidia-L4 is rejected"),
         ("--no-gpu-zonal-redundancy", "$0.672/hr, not $1.047: Rs 86,904 a month instead of Rs 109,854, for failover one instance never had"),
         ("--cpu 8 --memory 32Gi", "the instance the bill is for"),
         ("--max-instances 1", "each instance is a GPU; ten of them is a bill you will remember"),
         ("--min-instances 0", "the flag that makes the whole thing affordable, and the one people forget to restore"),
         ("--concurrency 4 --timeout 600", "a 4B model answers a few at a time; a cold start fits inside the timeout"),
         ("--no-allow-unauthenticated", "IAM: the gateway's account and the UI's, nobody else"),
         ("--labels slm-source=gguf|stock", "the image says what it serves; smoke-slm and the next cell read it back"),
         ("--startup-probe /api/tags", "200 means the model is listed, not merely that the process is alive")]
print()
for f, why in FLAGS:
    print(f"  {f:34} {why}")


In [ ]:
# Step four: the startup probe, because a 2 GB model is not instant.
#
# Cloud Run starts sending traffic when the container reports healthy. Ollama
# accepts connections before the model is loaded, so without a probe your first
# request arrives to a server that cannot answer it.
#
# /api/tags is the right endpoint: it lists loaded models, so a 200 means the
# model is actually there rather than merely that the process is alive.
PROBE_YAML = '''startupProbe:
  httpGet:
    path: /api/tags
    port: 8080
  initialDelaySeconds: 10
  periodSeconds: 5
  failureThreshold: 30        # 30 x 5s = 150s of patience'''
print(PROBE_YAML)
print()


def cold_start(model_gb: float) -> dict:
    """Where a cold start's seconds actually go. Rough, and rough is enough.

    The probe only measures the last row - everything above it happens before
    your container is even running, which is why "it started in 11 seconds"
    and "the user waited 40" are both true.
    """
    return {
        'GPU allocation':      5.0,
        'image pull (~2.5 GB)': 18.0,
        'container start':      5.0,
        'model load into VRAM': model_gb / 0.35,
    }


for gb in (2.0, 4.0, 9.0):
    parts = cold_start(gb)
    total = sum(parts.values())
    probe_sees = parts['model load into VRAM'] + parts['container start']
    print(f'  {gb:>4.1f} GB model -> user waits ~{total:>5.0f}s   '
          f'probe measures ~{probe_sees:>4.0f}s   '
          f'{"within budget" if probe_sees < 150 else "RAISE failureThreshold"}')

print()
for k, v in cold_start(2.0).items():
    print(f'    {k:24} {v:>5.0f}s')
print()
print('  A cold start on a GPU service is 30-60 seconds, not 300 milliseconds -')
print('  and most of it is the image, not the model.')
print('  That is the real cost of min-instances 0, and it is why the demo script')
print('  in 11.5 sends one warm-up request before anybody is watching.')


## Cell 6: Deployed, and the cold start timed


In [ ]:
# DEPLOYED, AND THE COLD START TIMED. make deploy-slm ran in Cloud Shell (the quota test is the deploy: a refusal costs
# nothing). /api/tags is the probe's endpoint and lists what is loaded; the first call after idle is the cold start -
# GPU allocation, the image pull, the process, the weights into VRAM - and the number is printed rather than
# estimated. Then Ollama's own door, and its OpenAI-compatible one: the door 11.1's clients and the gateway use.
svc = service("documind-slm")
assert svc, f"documind-slm is not deployed: make deploy-slm PROJECT={PROJECT_ID}  (or SLM_STOCK=gemma3:4b until 10.5's file lands)"
print(f"documind-slm: slm-source={svc['labels'].get('slm-source')} min-instances={svc['min_instances']} image={svc['image'].rsplit('/', 1)[-1]}")
assert svc["labels"].get("slm-source") == SLM_SOURCE_EXPECTED, f"the bucket says {SLM_SOURCE_EXPECTED} and the service says {svc['labels'].get('slm-source')}: redeploy"
status, tags, secs = slm("/api/tags", method="GET")
names = [m["name"] for m in tags.get("models", [])] if isinstance(tags, dict) else []
print(f"/api/tags in {secs:.1f}s (the cold start, when it was idle): {names}")
assert any(n.startswith("documind-slm") for n in names), (status, tags)
status, body, secs = slm("/api/generate", {"model": "documind-slm", "prompt": "Reply with the single word OK.", "stream": False})
print(f"/api/generate in {secs:.1f}s: {str(body.get('response', body) if isinstance(body, dict) else body).strip()[:60]!r}")
status, body, secs = slm("/v1/chat/completions", {"model": "documind-slm", "messages": [{"role": "user", "content": "Reply with the single word OK."}], "max_tokens": 10})
assert status == 200, (status, body)
print(f"/v1/chat/completions in {secs:.1f}s: {body['choices'][0]['message']['content'].strip()!r} - the same door 11.1's clients use")


## Cell 7: Behind the gateway, then behind the API
One completion through `documind-slm`; then the candidate revision on `MODEL_BACKEND=gateway` and the gate - 10.1's Cell 7 with a self-hosted model.


In [ ]:
from run_eval import live

# BEHIND THE GATEWAY, THEN BEHIND THE API. One completion through the gateway's documind-slm route: the response names
# what answered and the header carries the cost the gateway priced at the SLM's rate. Then the seam Module 10 built,
# with a self-hosted model in it: make candidate MODEL_BACKEND=gateway GENERATOR_MODEL=documind-slm tags a no-traffic
# revision of the SAME API image, whose generator sends the same SYSTEM, context and question to the gateway as a chat
# completion with a JSON response format, parses the same ModelDraft and resolves against the same packed chunks - and
# the gate judges it the way it judged the tuned Gemini in 10.1. Nothing above the API changed.
status, body, headers = gateway("documind-slm", "Reply with the single word OK.", max_tokens=10)
assert status == 200, (status, body)
print(f"gateway documind-slm -> served by {body.get('model')!r}, cost header {headers.get('x-litellm-response-cost')}")

def eval_line(url: str) -> int:
    os.environ["DOCUMIND_ID_TOKEN"] = id_token_as(MEMBER_SA, API_URL)          # the audience is the API's canonical URL (F42)
    os.environ["DOCUMIND_OUTSIDER_TOKEN"] = id_token_as(OUTSIDER_SA, API_URL)
    return live(url)

ver = requests.get(f"{CANDIDATE_URL}/version", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=60)
if ver.status_code == 200:
    v = ver.json()
    print(f"\ncandidate revision: model_backend={v['model_backend']} generator_model={v['generator_model']} git_sha={v['git_sha']}")
    assert v["model_backend"] == "gateway" and v["generator_model"] == "documind-slm", v
    rc = eval_line(CANDIDATE_URL)
    print("\nthe self-hosted candidate", "clears the gate" if rc == 0 else "does NOT clear the gate", "- the number is the decision; 10.4's judge says by how much")
else:
    print(f"\nno candidate revision (HTTP {ver.status_code}). In Cloud Shell:")
    print(f"  make candidate PROJECT={PROJECT_ID} MODEL_BACKEND=gateway GENERATOR_MODEL=documind-slm")
    print(f"  make eval-live PROJECT={PROJECT_ID} API={CANDIDATE_URL}")
    print(f"  make judge PROJECT={PROJECT_ID} API_B={CANDIDATE_URL}")


## Cell 8: Pin one tenant
A Firestore document the API reads before choosing a backend, and the usage row that proves it.


In [ ]:
from google.cloud import firestore

# PIN ONE TENANT. The residency customer's answers must come from the self-hosted model, and nobody else should pay
# for the GPU. On the lane that is a Firestore document, tenant_settings/{tenant}, the API reads once a minute before
# choosing a backend (main.py: tenant_settings / choose_for) - a field edit, not a redeploy - and it shows in the
# usage row's model_backend, which tenant_daily groups by. The cell pins the notebook's tenant, asks one question
# through retrieve(), reads the row back, and removes the pin: the lane stays on vertex.
api_env = service("documind-api")["env"]
assert api_env.get("LITELLM_URL"), "the live API image predates Module 11: make build deploy-services SERVICES_lean=api SCRIPTS_lean=commands/lesson-12.2.sh"
db = firestore.Client(project=PROJECT_ID)
pin = db.collection("tenant_settings").document(TENANT)
pin.set({"model_backend": "gateway", "generator_model": "documind-slm", "reason": "residency: inference on the self-hosted route", "set_by": "lesson 11.4"})
print(f"tenant_settings/{TENANT} = {pin.get().to_dict()}")
seen = None
try:
    for attempt in range(5):                       # the API caches a tenant's settings for 60 s; the log lags ~20 s
        got = documind_tools.retrieve("What is the per-trip cap on domestic travel reimbursement?", tenant_id=TENANT, brain="direct")
        time.sleep(25)
        rows = usage_rows(minutes=2, limit=5)
        seen = next((r for r in rows if r.get("model_backend") == "gateway"), None)
        print(f"  try {attempt + 1}: answerable={got.get('answerable')} rows={[(r.get('model_backend'), r.get('model')) for r in rows[:3]]}")
        if seen:
            break
finally:
    pin.delete()
    print(f"tenant_settings/{TENANT} removed: exists={pin.get().exists}")
assert seen, "no usage row said model_backend=gateway: is the gateway deployed, and is the API image the Module 11 one?"
print(f"\nthe row: tenant={seen['tenant']} model_backend={seen['model_backend']} model={seen['model']} cost_usd={seen['cost_usd']} latency_ms={seen['latency_ms']}")
print("per TENANT, not per environment: the bank gets in-country inference and nobody else pays for the GPU")


## Cell 9: The honest comparison
Retrieval once, two backends, four numbers - and the caveat that the per-1k figure is a property of how busy you are.


In [ ]:
import csv, io
import pandas as pd
from compare_backends import summarise, print_summary

# THE HONEST COMPARISON. Same questions, same corpus, same context: compare_backends.py retrieves ONCE per golden row
# through the one retrieve() and asks each backend to answer from that context through the gateway, so the only
# variable between rows is the model. Four numbers per backend - groundedness over answerable rows, citation precision
# per cited chunk, p95 latency, rupees per thousand queries - and the SLM's rupees are a RATE that only holds at the
# volume it was derived from. make compare does the same with 20 rows; make judge API_B= adds the pairwise judge.
env = {**os.environ, "LITELLM_URL": GATEWAY_URL, "LITELLM_ID_TOKEN": documind_tools._id_token(GATEWAY_URL), "RAG_API_URL": API_URL}
r = subprocess.run([sys.executable, f"{KIT}/deploy/services/slm/compare_backends.py", "--rows", str(ROWS), "--backends", "documind-general,documind-slm",
                    "--golden", f"{KIT}/deploy/evals/golden.jsonl"], capture_output=True, text=True, env=env)
if r.stderr.strip():
    print(r.stderr[-600:])
rows = list(csv.DictReader(io.StringIO(r.stdout)))
assert rows, f"no rows: exit {r.returncode}"
open("compare.csv", "w", encoding="utf-8").write(r.stdout)
summary = summarise(rows)
print_summary(summary)
assert "documind-slm" in summary and "documind-general" in summary
df = pd.DataFrame(rows)
df["grounded"] = df["grounded"].astype(int)
print("\ngrounded answers by shape (answerable rows):")
print(df[df["answerable"].isin(["True", "1"])].groupby(["backend", "shape"])["grounded"].mean().unstack().round(2).to_string())
print("\nthe column that decides it is the refusal shape: asked what the corpus cannot answer, which model said so?")


In [ ]:
# The caveat from cell 1, applied to the table you just printed.
L4_MONTH_INR = 1.42 * 24 * 30 * 85   # the INSTANCE (8vCPU+32GiB+L4), not the GPU line
GEM_PER_ANSWER = (800 * 1.50 + 200 * 7.50) / 1e6 * 85

print(f'{"answers/month":>15} {"self-hosted INR/1k":>20} {"gemini INR/1k":>15}  cheaper')
for n in (25_000, 100_000, 379_000, 500_000, 1_000_000):
    slm_per_1k = L4_MONTH_INR / n * 1000
    gem_per_1k = GEM_PER_ANSWER * 1000
    print(f'{n:>15,} {slm_per_1k:>20,.0f} {gem_per_1k:>15,.0f}  '
          f'{"self-hosted" if slm_per_1k < gem_per_1k else "gemini"}')

print()
print('One GPU, one price list, five completely different business cases.')
print('The per-1k figure for a self-hosted model is not a property of the model -')
print('it is a property of how busy you are. Publish it without the volume and')
print('you have published a number that is true for exactly one month.')


## Cell 10: Turn it off


In [ ]:
# TURN IT OFF. The most expensive thing in this course to forget: a warm L4 instance is Rs 86,904 a month for a service
# nobody is querying, it fails silently, and the bill arrives four weeks later while the service looks healthy. make
# slm-off sets min-instances back to 0; this cell reads the service and asserts it. The gate for the lesson is "the
# service is at min-instances 0 after the lesson", for exactly that reason.
svc = service("documind-slm")
print(f"documind-slm min-instances: {svc['min_instances']}")
for label, monthly in (("left at min-instances 1", L4_MONTH_INR), ("scaled to zero", 0.0)):
    print(f"  {label:26} Rs {monthly:>9,.0f}/month")
assert svc["min_instances"] == "0", f"make slm-off PROJECT={PROJECT_ID}  - now, not after the bill"
print(f"\nmake slm-off PROJECT={PROJECT_ID}   (and gateway-off, vllm-off: the end of every GPU day)")


## Where this goes
- **11.5** asks whether Cloud Run was the right home at all: it measures the lane's duty cycle from these usage rows and prices GKE against it.
- **10.4**'s judge explains what the gate decided: `make judge API_B=<candidate>` scores the self-hosted answers pairwise against the live revision's.

## ✅ Lesson 11.4 complete
- ✅ The idle-GPU bill, the break-even and the Mumbai table, before anything was deployed
- ✅ The Modelfile generated from the tokenizer, read back beside the GGUF; the placeholder named as a bug
- ✅ The kit's image and `make deploy-slm` flag by flag; the cold start timed on the real service
- ✅ One completion through the gateway's route; the candidate on `MODEL_BACKEND=gateway` judged by the gate
- ✅ One tenant pinned by a document and read back off the usage row's `model_backend`
- ✅ The honest comparison live; the service asserted at min-instances 0
